# Edge sensitivity and Shapley results

This notebook reads the fixed-weight edge ablation and sampled Shapley outputs for the `replace_000` and `replace_100` models.

- **Edge removal**: disables one existing C?C connection, with the trained parameters held fixed. The drop is baseline metric minus the metric after removal.
- **Shapley**: averages each edge's marginal change in the chosen metric over sampled edge orderings.
- **Retraining selections**: ranks validation-set removals using `SELECTION_METRIC`, then writes the top N edges for each model to `selected_edges_for_retraining.csv`. The final test split is displayed for reporting, not used to choose edges.

Change `DATASET` to `celeba` or `cfmnist` and adjust `SELECTION_METRIC` below.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiments").exists() and (PROJECT_ROOT.parent / "experiments").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET = "celeba"  # "celeba" or "cfmnist"
ANALYSIS_CONFIG = PROJECT_ROOT / "all_configs" / "analysis" / "edge_sensitivity" / f"{DATASET}_replace_000_vs_replace_100.yaml"
analysis_config = yaml.safe_load(ANALYSIS_CONFIG.read_text(encoding="utf-8"))
SELECTION_METRIC = analysis_config.get("selection_metric", "task_accuracy")  # can override here, e.g. concept_accuracy_003
TOP_N = int(analysis_config.get("top_edges_per_model", 3))
RESULTS_DIR = PROJECT_ROOT / "experiments" / "edge_sensitivity" / DATASET
SELECTION_CSV = RESULTS_DIR / "selected_edges_for_retraining.csv"

coalitions = pd.read_csv(RESULTS_DIR / "coalition_metrics.csv")
shapley = pd.read_csv(RESULTS_DIR / "shapley_values.csv")
edges = pd.read_csv(RESULTS_DIR / "edge_list.csv")

# Reproducible paper-style exports for edge sensitivity and Shapley analysis.
from matplotlib.colors import LinearSegmentedColormap
REPORT_DIR = PROJECT_ROOT / "report"
CSV_REPORT_DIR = REPORT_DIR / "a_metric_sens_csv"
FIGURE_REPORT_DIR = REPORT_DIR / "a_metric_sens_figures"
CSV_REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
LATEX_REPORT_PATH = REPORT_DIR / "a_metric_sens_latex.tex"
LATEX_FIGURES = []
PAPER_CMAP = LinearSegmentedColormap.from_list(
    "blue_grey_purple", ["#3B6FB6", "#D8D9DE", "#6542A6"]
)
EDGE_PALETTE = [
    "#6542A6", "#3B6FB6", "#737782", "#8B6CC1", "#5793CC", "#9A9DA5",
    "#4D3A80", "#2D5F91", "#B2B4BA", "#A084CC", "#78A9D6", "#5C626E",
    "#7B63A8", "#466F9F", "#858A94", "#B39AD8", "#91B9DD", "#C4C6CB",
]
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "axes.edgecolor": "#555963",
    "text.color": "#292A30",
    "axes.labelcolor": "#292A30",
    "xtick.color": "#41434B",
    "ytick.color": "#41434B",
})

def save_paper_figure(fig, stem, caption):
    pdf_path = FIGURE_REPORT_DIR / f"{stem}.pdf"
    png_path = FIGURE_REPORT_DIR / f"{stem}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight", dpi=300)
    LATEX_FIGURES[:] = [item for item in LATEX_FIGURES if item["stem"] != stem]
    LATEX_FIGURES.append({"stem": stem, "pdf": pdf_path.relative_to(PROJECT_ROOT).as_posix(), "caption": caption})
    return pdf_path, png_path

def write_latex_report():
    blocks = [
        "% Auto-generated by notebook/edge_sensitivity_shapley_analysis.ipynb.",
        "% Compile from the repository root; requires \\usepackage{graphicx}.",
        "",
    ]
    for item in LATEX_FIGURES:
        label = "fig:" + item["stem"].replace("_", "-")
        blocks.extend([
            "\\begin{figure}[t]", "\\centering",
            f"\\includegraphics[width=\\linewidth]{{{item['pdf']}}}",
            f"\\caption{{{item['caption']}}}", f"\\label{{{label}}}",
            "\\end{figure}", "",
        ])
    LATEX_REPORT_PATH.write_text("\n".join(blocks) + "\n", encoding="utf-8")
    print("Wrote LaTeX figure snippet:", LATEX_REPORT_PATH)

print("Loaded", DATASET, "from", RESULTS_DIR)
print("Models:", sorted(edges.model.unique()))
print("Available metric examples:", sorted(coalitions.metric.unique())[:30])
# Original CREAM run used as a reference (from the existing topology-density analysis).
ORIGINAL_RUNS = {
    "celeba": PROJECT_ROOT / "experiments/CelebA/train_cbm/Standard_CelebA/CREAM/CREAM_best_celeba",
    "cfmnist": PROJECT_ROOT / "experiments/Complete_Concept_FMNIST/train_cbm/Standard_FashionMNIST/CREAM/CREAM_best_cfmnist",
}
ORIGINAL_RUN_ROOT = ORIGINAL_RUNS[DATASET]
_original_metric_files = sorted(ORIGINAL_RUN_ROOT.glob("**/last_metrics/*.csv")) if ORIGINAL_RUN_ROOT.exists() else []
ORIGINAL_METRICS_PATH = _original_metric_files[-1] if _original_metric_files else None
_original_intervention_files = sorted(ORIGINAL_RUN_ROOT.glob("**/intervention_results.csv")) if ORIGINAL_RUN_ROOT.exists() else []
ORIGINAL_INTERVENTIONS_PATH = _original_intervention_files[-1] if _original_intervention_files else None
print("Original CREAM reference:", ORIGINAL_RUN_ROOT)
print("Original CREAM metrics:", ORIGINAL_METRICS_PATH or "not found")
print("Original CREAM intervention CSV:", ORIGINAL_INTERVENTIONS_PATH or "not found")


## 1. Baseline predictions and intervention performance\n\nThe table compares the two topology models; the original CREAM test metrics and saved intervention curve are added when their result files are present. The next plot shows test task accuracy as concepts are intervened on. This intervention curve is separate from the edge-removal heatmap below.\n

In [ ]:
baseline = coalitions[(coalitions.coalition_type == "baseline") & (coalitions.split.isin(["val", "test"]))].copy()
base_pivot = baseline.pivot_table(index=["metric"], columns=["model", "split"], values="metric_value")
display(base_pivot.round(4))
base_pivot.reset_index().to_csv(CSV_REPORT_DIR / "baseline_topology_metrics.csv", index=False)

# Original CREAM has no edge-removal rows, so compare it as a baseline reference.
if ORIGINAL_METRICS_PATH is not None:
    original_row = pd.read_csv(ORIGINAL_METRICS_PATH).iloc[0]
    original_values = {"task_accuracy": original_row.get("test_task_accuracy"),
                       "concept_accuracy": original_row.get("test_concept_accuracy")}
    original_table = pd.DataFrame({"Original CREAM (test)": original_values})
    print("Original CREAM baseline metrics (from its saved last_metrics CSV):")
    display(original_table.round(4))
    original_table.rename_axis("metric").reset_index().to_csv(CSV_REPORT_DIR / "baseline_original_cream_metrics.csv", index=False)

# Plot test task-accuracy intervention curves for replace_000 and replace_100.
# Add the curve saved by the original CREAM run when that file is present.
curve = baseline[baseline.metric.str.startswith("intervention_task_accuracy_") & (baseline.split == "test")].copy()
if not curve.empty:
    parts = curve.metric.str.extract(r"intervention_task_accuracy_(individual|group)_(\d+)")
    curve["intervention_mode"] = parts[0]
    curve["num_interventions"] = parts[1].astype(int)

original_curve = pd.DataFrame()
if ORIGINAL_INTERVENTIONS_PATH is not None:
    original_curve = pd.read_csv(ORIGINAL_INTERVENTIONS_PATH)
    if not original_curve.empty and "num_interventions" in original_curve:
        task_col = next((c for c in ["test_task_accuracy", "task_accuracy", "test_accuracy", "accuracy"] if c in original_curve), None)
        if task_col:
            original_curve["curve_accuracy"] = pd.to_numeric(original_curve[task_col], errors="coerce")
            original_curve["intervention_mode"] = original_curve.get("group_interventions", False).map(
                lambda x: "group" if str(x).strip().lower() in ("true", "1") else "individual"
            ) if hasattr(original_curve.get("group_interventions", False), "map") else "individual"

curve_export_rows = []
if not curve.empty:
    curve_export_rows.extend(curve[["model", "split", "intervention_mode", "num_interventions", "metric_value"]]
                             .rename(columns={"metric_value": "task_accuracy"}).to_dict("records"))
if not original_curve.empty and "curve_accuracy" in original_curve:
    curve_export_rows.extend(original_curve[["intervention_mode", "num_interventions", "curve_accuracy"]]
                             .rename(columns={"curve_accuracy": "task_accuracy"})
                             .assign(model="Original CREAM", split="test")
                             [["model", "split", "intervention_mode", "num_interventions", "task_accuracy"]].to_dict("records"))
pd.DataFrame(curve_export_rows).to_csv(CSV_REPORT_DIR / "test_intervention_curves.csv", index=False)

modes = sorted(set(curve.intervention_mode.dropna().unique()) | set(original_curve.intervention_mode.dropna().unique())) if not curve.empty or not original_curve.empty else []
for mode in modes:
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    current = curve[curve.intervention_mode == mode] if not curve.empty else pd.DataFrame()
    for model, frame in current.groupby("model"):
        ax.plot(frame.num_interventions, frame.metric_value, marker="o", label=model)
    if not original_curve.empty:
        old = original_curve[original_curve.intervention_mode == mode].copy()
        if not old.empty:
            ax.plot(old.num_interventions, old.curve_accuracy, marker="s", linestyle="--", color="black", label="Original CREAM")
    ax.set(title=f"Test task accuracy under {mode} interventions", xlabel="Number of intervened concepts", ylabel="Task accuracy")
    ax.set_ylim(0, 1)
    ax.legend()
    save_paper_figure(fig, f"{DATASET}_intervention_task_accuracy_{mode}",
                      f"Test task accuracy under {mode} interventions for Graph A, Graph B, and original CREAM when available ({DATASET}).")
    plt.show()


## 2. Which existing edges matter when weights are frozen?

Each row below is the change from the complete graph to a model with one edge removed. Positive drops mean the removal hurt that metric; negative values mean the metric improved after removal. Validation drops are used to choose edges for retraining.

In [ ]:
base = coalitions[(coalitions.coalition_type == "baseline")].set_index(["model", "split", "metric"]).metric_value
abl = coalitions[coalitions.coalition_type == "single_edge_removed"].copy()
abl["baseline_value"] = [base.loc[(r.model, r.split, r.metric)] for r in abl.itertuples()]
abl["metric_drop_from_removal"] = abl.baseline_value - abl.metric_value
edge_effects = abl.copy()
validation_effects = edge_effects[edge_effects.split == "val"].copy()
edge_effects.to_csv(CSV_REPORT_DIR / "edge_sensitivity_all_metrics.csv", index=False)

# Human-readable edge names use the graph's source and target concept labels.
validation_effects["edge_label"] = (
    validation_effects.source_concept.astype(str).str.replace("_", " ", regex=False)
    + " to "
    + validation_effects.target_concept.astype(str).str.replace("_", " ", regex=False)
)
selected_view = validation_effects[validation_effects.metric == SELECTION_METRIC].sort_values(
    "metric_drop_from_removal", ascending=False
)
for model, frame in selected_view.groupby("model"):
    print(f"{model} ranked by validation {SELECTION_METRIC}")
    display(frame[["edge_label", "baseline_value", "metric_value", "metric_drop_from_removal"]]
            .head(15).round(4))

# A compact A-over-B heatmap. Each cell is validation baseline accuracy
# minus accuracy after removing that edge. Intervention-at-each-budget metrics
# are shown here as rows; their full accuracy curves are plotted in section 1.
model_order = [m for m in ["replace_000", "replace_100"] if m in validation_effects.model.unique()]
metric_order = list(validation_effects.metric.drop_duplicates())
metric_order = sorted(metric_order, key=lambda m: (
    0 if m == "task_accuracy" else 1 if m == "concept_accuracy" else
    2 if m.startswith("concept_accuracy_") else
    3 if m.startswith("intervention_task_auc") else
    4 if m.startswith("intervention_concept_auc") else 5, m
))
concept_by_index = {}
for r in edges[["source_index", "source_concept"]].drop_duplicates().itertuples(index=False):
    concept_by_index[int(r.source_index)] = str(r.source_concept).replace("_", " ")
for r in edges[["target_index", "target_concept"]].drop_duplicates().itertuples(index=False):
    concept_by_index[int(r.target_index)] = str(r.target_concept).replace("_", " ")

def pretty_metric(metric):
    if metric == "task_accuracy": return "Task accuracy"
    if metric == "concept_accuracy": return "Mean concept accuracy"
    m = __import__("re").fullmatch(r"concept_accuracy_(\d+)", metric)
    if m: return "Concept accuracy: " + concept_by_index.get(int(m.group(1)), f"Concept {m.group(1)}")
    m = __import__("re").fullmatch(r"intervention_(task|concept)_accuracy_(individual|group)_(\d+)", metric)
    if m:
        target, mode, count = m.groups()
        target = "task" if target == "task" else "mean concept"
        return f"{target.title()} after {mode} intervention: {int(count)} concepts"
    m = __import__("re").fullmatch(r"intervention_(task|concept)_auc_(individual|group)", metric)
    if m: return f"{m.group(1).title()} intervention AUC ({m.group(2)})"
    return metric.replace("_", " ").title()

edge_labels_by_model = {}
for model in model_order:
    edge_labels_by_model[model] = (validation_effects[validation_effects.model == model]
                                   [["edge_label", "source_index", "target_index"]]
                                   .drop_duplicates().sort_values(["source_index", "target_index"])
                                   .edge_label.tolist())
all_drops = validation_effects.metric_drop_from_removal.to_numpy(dtype=float)
color_limit = float(np.nanmax(np.abs(all_drops))) if len(all_drops) else 1.0
color_limit = color_limit or 1.0
max_edges = max((len(v) for v in edge_labels_by_model.values()), default=1)
fig_height = max(8, 0.25 * len(metric_order) * len(model_order) + 4)
fig = plt.figure(figsize=(max(14, 0.85 * max_edges), fig_height), layout="constrained")
grid = fig.add_gridspec(len(model_order), 2, width_ratios=[1, 0.035], hspace=0.30, wspace=0.04)
axes = [fig.add_subplot(grid[i, 0]) for i in range(len(model_order))]
cax = fig.add_subplot(grid[:, 1])
image = None
for ax, model in zip(axes, model_order):
    frame = validation_effects[validation_effects.model == model]
    edge_order = edge_labels_by_model[model]
    matrix = (frame.pivot_table(index="metric", columns="edge_label", values="metric_drop_from_removal", aggfunc="mean")
              .reindex(index=metric_order, columns=edge_order))
    image = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=PAPER_CMAP,
                      vmin=-color_limit, vmax=color_limit, interpolation="nearest")
    graph_label = "Graph A" if model == "replace_000" else "Graph B"
    ax.set_title(f"{graph_label} -- {model}", loc="left", fontweight="bold", pad=10)
    ax.set_xticks(np.arange(len(edge_order)))
    labels = [label.replace(" to ", "\nto ") for label in edge_order]
    ax.set_xticklabels(labels, rotation=55, ha="right", rotation_mode="anchor", fontsize=8)
    ax.set_yticks(np.arange(len(metric_order)))
    ax.set_yticklabels([pretty_metric(m) for m in metric_order], fontsize=8)
    ax.set_xlabel("Removed edge: source concept to target concept")
    ax.set_ylabel("Validation metric")
    if len(metric_order) <= 35 and len(edge_order) <= 14:
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                value = matrix.iat[i, j]
                if pd.notna(value):
                    text_color = "white" if abs(value) > color_limit * 0.48 else "black"
                    ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=6.5, color=text_color)
fig.colorbar(image, cax=cax, label="Baseline minus after edge removal (positive = removal hurts)")
fig.suptitle("Validation edge sensitivity: all recorded metrics", fontsize=14)
save_paper_figure(fig, f"{DATASET}_edge_sensitivity_all_metrics",
                  f"Change in validation metrics after removing each allowed concept-to-concept edge for Graph A and Graph B ({DATASET}); positive values mean removal reduced performance.")
plt.show()


## 3. Shapley contributions across all metrics\n\nThe table ranks edges for the configured selection metric. Below it, two horizontal plots show all recorded metrics: Graph A (`replace_000`) and Graph B (`replace_100`). Each row is a metric; each colored dot is an edge identified by its source and target concept names. The x-position is the edge's Shapley value, and the whisker is its permutation standard error. Positive values mean that enabling the edge improves that metric on average across sampled coalitions.\n

In [ ]:
# Keep the ranked table for the configured selection metric, then plot every recorded metric.
shapley.to_csv(CSV_REPORT_DIR / "shapley_all_metrics.csv", index=False)
shapley_metric = SELECTION_METRIC
shapley_part = shapley[shapley.metric == shapley_metric].copy()
for model, frame in shapley_part.groupby("model"):
    frame = frame.sort_values("shapley_value", ascending=False)
    print(f"{model} ranked by Shapley value for {shapley_metric}")
    display(frame[["source_concept", "target_concept", "shapley_value", "permutation_se", "n_permutations"]].head(20).round(5))

# Readable names for metrics, including individual-concept accuracy and intervention metrics.
concept_by_index = {}
for r in edges[["source_index", "source_concept"]].drop_duplicates().itertuples(index=False):
    concept_by_index[int(r.source_index)] = str(r.source_concept).replace("_", " ")
for r in edges[["target_index", "target_concept"]].drop_duplicates().itertuples(index=False):
    concept_by_index[int(r.target_index)] = str(r.target_concept).replace("_", " ")

def shapley_metric_label(metric):
    if metric == "task_accuracy": return "Task accuracy"
    if metric == "concept_accuracy": return "Mean concept accuracy"
    m = __import__("re").fullmatch(r"concept_accuracy_(\d+)", metric)
    if m: return "Concept accuracy: " + concept_by_index.get(int(m.group(1)), f"Concept {m.group(1)}")
    m = __import__("re").fullmatch(r"intervention_(task|concept)_accuracy_(individual|group)_(\d+)", metric)
    if m:
        target, mode, count = m.groups()
        target = "Task" if target == "task" else "Mean concept"
        return f"{target} accuracy after {mode} intervention ({int(count)} concepts)"
    m = __import__("re").fullmatch(r"intervention_(task|concept)_auc_(individual|group)", metric)
    if m:
        target, mode = m.groups()
        return f"{target.title()} intervention AUC ({mode})"
    return metric.replace("_", " ").title()

metric_order = list(shapley.metric.drop_duplicates())
metric_order = sorted(metric_order, key=lambda m: (
    0 if m == "task_accuracy" else 1 if m == "concept_accuracy" else
    2 if m.startswith("concept_accuracy_") else
    3 if m.startswith("intervention_task_auc") else
    4 if m.startswith("intervention_concept_auc") else 5, m
))

# One separate horizontal plot per graph. Each colored dot is one edge's Shapley
# value for that metric; horizontal whiskers show permutation standard error.
for model in ["replace_000", "replace_100"]:
    frame = shapley[shapley.model == model].copy()
    if frame.empty:
        print(f"No Shapley rows found for {model}")
        continue
    frame["edge_label"] = (frame.source_concept.astype(str).str.replace("_", " ", regex=False)
                           + " to "
                           + frame.target_concept.astype(str).str.replace("_", " ", regex=False))
    edge_order = (frame[["edge_label", "source_index", "target_index"]].drop_duplicates()
                  .sort_values(["source_index", "target_index"]).edge_label.tolist())
    palette = [EDGE_PALETTE[i % len(EDGE_PALETTE)] for i in range(len(edge_order))]
    fig_height = max(7, 0.48 * len(metric_order) + 2)
    fig, ax = plt.subplots(figsize=(14, fig_height), layout="constrained")
    offsets = np.linspace(-0.32, 0.32, len(edge_order)) if len(edge_order) > 1 else np.array([0.0])
    for edge_idx, edge_label in enumerate(edge_order):
        edge_rows = frame[frame.edge_label == edge_label]
        y_by_metric = {metric: i for i, metric in enumerate(metric_order)}
        y = np.array([y_by_metric[m] + offsets[edge_idx] for m in edge_rows.metric])
        ax.errorbar(edge_rows.shapley_value.to_numpy(dtype=float), y,
                    xerr=edge_rows.permutation_se.to_numpy(dtype=float),
                    fmt="o", markersize=5, capsize=2, linewidth=0.9,
                    color=palette[edge_idx], label=edge_label)
    ax.axvline(0, color="black", linewidth=0.9)
    ax.set_yticks(np.arange(len(metric_order)))
    ax.set_yticklabels([shapley_metric_label(m) for m in metric_order])
    ax.invert_yaxis()
    ax.set_xlabel("Shapley value (average accuracy gain from enabling the edge)")
    graph_label = "Graph A" if model == "replace_000" else "Graph B"
    ax.set_title(f"{graph_label} ({model}): Shapley values for all metrics", loc="left", fontweight="bold")
    ax.grid(axis="x", alpha=0.25)
    ax.legend(title="Edge: source to target concept", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    save_paper_figure(fig, f"{DATASET}_shapley_all_metrics_{model}",
                      f"Shapley values across all recorded metrics for {graph_label} ({model}) on {DATASET}. Points identify source-to-target concept edges; horizontal whiskers show permutation standard error.")
    plt.show()

write_latex_report()
print("Saved CSV tables under:", CSV_REPORT_DIR)
print("Saved PDF and PNG figures under:", FIGURE_REPORT_DIR)


## 4. Export the top removal candidates for retraining

Selection is done from **validation edge-removal drops**. Choose the criterion above before exporting. The output contains the top `TOP_N` edges per model; the generator turns each row into a DAG with that edge removed and a matching fixed-backbone CREAM training config.

In [ ]:
if selected_view.empty:
    raise ValueError(f"No validation rows found for SELECTION_METRIC={SELECTION_METRIC!r}")
selected = (selected_view.sort_values(["model", "metric_drop_from_removal"], ascending=[True, False])
            .groupby("model", as_index=False).head(TOP_N).copy())
selected["selection_metric"] = SELECTION_METRIC
selected["selection_rank"] = selected.groupby("model")["metric_drop_from_removal"].rank(method="first", ascending=False).astype(int)
selected = selected[["model", "edge_id", "source_index", "target_index", "source_concept", "target_concept", "selection_metric", "selection_rank", "metric_drop_from_removal"]]
selected.to_csv(SELECTION_CSV, index=False)
print("Saved retraining selections:", SELECTION_CSV)
display(selected)

## Exported paper assets

Running the notebook saves analysis CSVs to `report/a_metric_sens_csv/`, PDF and PNG figures to `report/a_metric_sens_figures/`, and generates `report/a_metric_sens_latex.tex` with figure environments. The plots use a serif paper-style font and a purple, blue, and grey palette.
